# Health PSA - Full Pipeline (Extract + Find Sources)

This merges two stages into one notebook, so you only ever upload one file: `PSA_KE_Final.csv`.

1. **Extract** Health-domain rows from `PSA_KE_Final.csv` into the project's required schema
2. **Search** for a real source link per PSA (via SerpAPI), backfilling the `Source` column

**Run the cells in order, top to bottom - do not skip the upload step.**

## Step 1: Install dependencies

In [ ]:
!pip install -q serpapi google-search-results tqdm

  Preparing metadata (setup.py) ... done


## Step 2: Upload PSA_KE_Final.csv

**Run this cell now and select your `PSA_KE_Final.csv` file before continuing.** If you skip this, Step 3 will fail with a clear "file not found" message telling you to come back here.

In [ ]:
from google.colab import files
uploaded = files.upload()  # select PSA_KE_Final.csv

Saving PSA_KE_Final.csv to PSA_KE_Final.csv


## Step 3: Extract Health-domain rows into the required schema

In [ ]:
import pandas as pd
from pathlib import Path

INPUT_FILE = "PSA_KE_Final.csv"          # upload this in Step 2 below
OUTPUT_FILE = "health_psas_from_class_dataset.csv"
DOMAIN = "Health"
PREFIX = "HEALTH"

# ---------------------------------------------------------------------------
# Same validation as extract_psa.py - clear errors instead of a confusing
# FileNotFoundError deep inside pandas.
# ---------------------------------------------------------------------------
if not Path(INPUT_FILE).exists():
    raise SystemExit(
        f"'{INPUT_FILE}' not found. Run the upload cell in Step 2 above and "
        f"select your PSA_KE_Final.csv file, THEN come back and re-run this cell."
    )

df = pd.read_csv(INPUT_FILE, dtype=str)

required_source_cols = ["PSA_Id", "Domain", "English", "Kiswahili"]
missing = [c for c in required_source_cols if c not in df.columns]
if missing:
    raise SystemExit(
        f"Expected column(s) {missing} not found in {INPUT_FILE}. "
        f"Columns actually present: {list(df.columns)}"
    )

available_domains = sorted(df["Domain"].dropna().unique())
if DOMAIN not in available_domains:
    raise SystemExit(f"Domain '{DOMAIN}' not found. Available domains: {available_domains}")

subset = df[df["Domain"] == DOMAIN].copy()

rows = []
skipped = 0
for _, r in subset.iterrows():
    english = str(r.get("English", "")).strip()
    kiswahili = str(r.get("Kiswahili", "")).strip()
    if english.lower() == "nan":
        english = ""
    if kiswahili.lower() == "nan":
        kiswahili = ""

    if not english and not kiswahili:
        skipped += 1
        continue  # never invent content for a row with nothing usable

    rows.append({
        "PSA_ID": f"{PREFIX}_{len(rows) + 1:03d}",
        "Domain": DOMAIN,
        "Sub_Category": "",
        "English": english,
        "Kiswahili": kiswahili,
        "Target_Language": "",
        "Source": "PSA_KE_Final.csv (class-provided dataset)",
        "Date": "",
        "Metadata": f"original_id={r.get('PSA_Id', '')}; class=extracted_from_class_dataset",
    })

out_df = pd.DataFrame(rows, columns=[
    "PSA_ID", "Domain", "Sub_Category", "English", "Kiswahili",
    "Target_Language", "Source", "Date", "Metadata",
])
out_df.to_csv(OUTPUT_FILE, index=False)

print(f"Extracted {len(out_df)} rows for domain '{DOMAIN}' -> {OUTPUT_FILE}")
print(f"Skipped {skipped} row(s) with no usable text in either language")
print("\nProceed to Step 4 below to search for real source links for these rows.")


Extracted 495 rows for domain 'Health' -> health_psas_from_class_dataset.csv
Skipped 0 row(s) with no usable text in either language

Proceed to Step 4 below to search for real source links for these rows.


## Step 4: Search for real source links

You'll be asked for a SerpAPI key (get one free at https://serpapi.com/users/sign_up, then copy it from https://serpapi.com/manage-api-key). This step is optional - your dataset from Step 3 is already usable without it.

In [ ]:
import time
import pandas as pd
from serpapi import GoogleSearch
from tqdm import tqdm
from getpass import getpass

# ---------------------------------------------------------------------------
# Same fixes as the Agriculture version:
#   - API key entered securely, never hardcoded
#   - SerpAPI errors detected and printed clearly (not silently swallowed)
#   - Missing file/columns raise a clear, actionable message
#   - Progress bar + politeness delay between requests
#
# WHAT THIS DOES DIFFERENTLY FOR HEALTH: your Health rows (extracted from
# PSA_KE_Final.csv) don't have a real per-row source URL yet - just a
# generic placeholder ("PSA_KE_Final.csv (class-provided dataset)"). This
# cell searches Google (via SerpAPI) for each PSA's likely real source page,
# biased toward Kenyan government/health sites, and backfills the Source
# column with a real URL when a plausible match is found.
#
# HONEST LIMITATION: a search result matching the PSA's wording is a
# PLAUSIBLE source, not a confirmed one - spot-check a sample of the found
# links yourself before treating them as verified citations, same as every
# other extraction method used in this project.
# ---------------------------------------------------------------------------

API_KEY = getpass("Enter your SerpAPI key: ")

INPUT_FILE = "health_psas_from_class_dataset.csv"   # output of extract_psa.py
OUTPUT_FILE = "health_psas_with_sources.csv"
REQUEST_DELAY_SECONDS = 1.5
MAX_QUERY_CHARS = 100          # keep queries short - long queries return worse matches
# Bias the search toward genuine Kenyan government/health sources rather than
# generic results - adjust this if you want a different bias (e.g. a specific
# domain like "site:health.go.ke").
QUERY_CONTEXT = "Kenya Ministry of Health"


def get_serp_link(query: str) -> str | None:
    """Look up the top Google result link for a query via SerpAPI.
    Returns None (not a crash) if the API errors, the key is invalid/out
    of quota, or no organic result is found."""
    params = {"engine": "google", "q": query, "api_key": API_KEY}
    try:
        search = GoogleSearch(params)
        results = search.get_dict()
    except Exception as e:
        print(f"  [SerpAPI request failed] query={query[:60]!r} -> {type(e).__name__}: {e}")
        return None

    if "error" in results:
        # SerpAPI returns HTTP 200 with an "error" field for bad keys/quota
        # issues rather than raising an exception - must check explicitly.
        print(f"  [SerpAPI returned an error] {results['error']} (query: {query[:60]!r})")
        return None

    organic = results.get("organic_results", [])
    if not organic:
        return None
    return organic[0].get("link")


# ---------------------------------------------------------------------------
# Validate input file and required columns before looping.
# ---------------------------------------------------------------------------
try:
    df = pd.read_csv(INPUT_FILE, dtype=str)
except FileNotFoundError:
    raise SystemExit(
        f"'{INPUT_FILE}' not found. In Colab, upload it via the Files panel "
        f"on the left (folder icon), or run:\n"
        f"    from google.colab import files\n"
        f"    files.upload()\n"
        f"then re-run this cell. (This should be the output of extract_psa.py "
        f"run against PSA_KE_Final.csv with --domain Health.)"
    )

required_columns = ["English"]
missing = [c for c in required_columns if c not in df.columns]
if missing:
    raise SystemExit(
        f"Expected column(s) {missing} not found in {INPUT_FILE}. "
        f"Columns actually present: {list(df.columns)}."
    )

if "Source" not in df.columns:
    df["Source"] = ""

# ---------------------------------------------------------------------------
# Search for a real source link per row, using the PSA's English text
# (truncated) plus a Kenya health context bias.
# ---------------------------------------------------------------------------
new_sources = []
found_count = 0

for _, row in tqdm(df.iterrows(), total=len(df), desc="Searching for real source links"):
    english_text = str(row.get("English", "")).strip()
    if not english_text or english_text.lower() == "nan":
        new_sources.append(row.get("Source", ""))
        continue

    query = f"{QUERY_CONTEXT} {english_text[:MAX_QUERY_CHARS]}"
    link = get_serp_link(query)
    time.sleep(REQUEST_DELAY_SECONDS)

    if link:
        new_sources.append(link)
        found_count += 1
    else:
        # Keep the original placeholder rather than leaving Source blank -
        # never invent a source, and don't lose the fact this row came
        # from the class dataset just because a web search didn't confirm it.
        new_sources.append(row.get("Source", ""))

df["Source"] = new_sources

print(f"\nDone. Found a plausible real source link for {found_count}/{len(df)} rows.")
print("Spot-check a sample of these links yourself before treating them as confirmed citations.")

df.to_csv(OUTPUT_FILE, index=False)
print(f"Saved: {OUTPUT_FILE}")


Enter your SerpAPI key: ··········


Searching for real source links:  51%|█████     | 251/495 [28:07<29:06,  7.16s/it]

  [SerpAPI returned an error] Your account has run out of searches. (query: 'Kenya Ministry of Health Mental health is the state of wellb')


Searching for real source links:  51%|█████     | 252/495 [28:09<22:28,  5.55s/it]

  [SerpAPI returned an error] Your account has run out of searches. (query: 'Kenya Ministry of Health Mothers should return for postnatal')


...

  [SerpAPI returned an error] Your account has been throttled. You are exceeding 250 searches per hour. Please upgrade your plan, spread out your searches, or contact support. (query: 'Kenya Ministry of Health Self-care is important - it directl')


Searching for real source links: 100%|█████████▉| 494/495 [35:22<00:01,  1.80s/it]

  [SerpAPI returned an error] Your account has been throttled. You are exceeding 250 searches per hour. Please upgrade your plan, spread out your searches, or contact support. (query: 'Kenya Ministry of Health Climate change affects mental healt')


Searching for real source links: 100%|██████████| 495/495 [35:24<00:00,  4.29s/it]


Done. Found a plausible real source link for 251/495 rows.
Spot-check a sample of these links yourself before treating them as confirmed citations.
Saved: health_psas_with_sources.csv


## Step 5: Preview the result

Check a sample of the `Source` column before trusting the links as confirmed citations.

In [ ]:
import pandas as pd
result = pd.read_csv('health_psas_with_sources.csv')
result[['PSA_ID', 'English', 'Source']].head(10)

,PSA_ID,English,Source
0,HEALTH_001,We are advising all health workers to ensure t...,https://www.instagram.com/reel/DZfhX7ys8F1/
1,HEALTH_002,Restaurants must;\n1.ensure quality & safety o...,https://www.facebook.com/kebs.org/posts/press-...
2,HEALTH_003,We are advising members of the public to put o...,https://www.facebook.com/KTNNewsKenya/posts/he...
3,HEALTH_004,This are the first steps to keeping your famil...,https://www.facebook.com/MinistryofHealthTT/po...
4,HEALTH_005,Alcohol and Smoking will increase the risks of...,https://en.wikipedia.org/wiki/Kenya
5,HEALTH_006,Be safe from coronavirus.Washing your hands is...,https://www.facebook.com/UNICEFKenya/posts/one...
6,HEALTH_007,How to boost your immune system against the CO...,https://www.researchgate.net/publication/35223...
7,HEALTH_008,You have to be registered to get the COVID-19 ...,https://www.facebook.com/SpokesPersonKenya/pos...
8,HEALTH_009,Health CS Mutahi Kagwe says total lockdown on ...,https://www.facebook.com/nation/posts/coming-n...
9,HEALTH_010,"Covid-19 variant from India detected in Kenya,...",https://pmc.ncbi.nlm.nih.gov/articles/PMC8902869/


## Step 6: Download the result

In [ ]:
from google.colab import files
files.download('health_psas_with_sources.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>